# Besoin Client 3 - Systeme d'alerte tempete

Ce notebook prepare les donnees de `export_IA.csv`, entraine un modele de classification sur la cible `fk_arb_etat`, evalue les performances, puis exporte le modele final au format `.pkl`.

L'alerte tempete est derivee de la prediction de `fk_arb_etat` : un arbre est considere en alerte si l'etat predit n'est pas `en place`.

Le livrable final genere ici est `bc3_model.pkl`, directement chargeable par `predict_bc3.py`.


In [ ]:
!pip -q install scikit-learn pandas matplotlib seaborn


In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


In [ ]:
CANDIDATE_PATHS = [
    Path('/content/export_IA.csv'),
    Path('/content/content/export_IA.csv'),
    Path('export_IA.csv'),
    Path('content/export_IA.csv'),
]

DATA_PATH = next((path for path in CANDIDATE_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'export_IA.csv introuvable. Emplacements testes : ' + ', '.join(str(p) for p in CANDIDATE_PATHS)
    )

OUTPUT_DIR = Path('/content/models')
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Fichier source detecte : {DATA_PATH}')

TARGET_SOURCE = 'fk_arb_etat'
SAFE_STATE = 'en place'
FEATURE_COLUMNS = [
    'X',
    'Y',
    'clc_quartier',
    'clc_secteur',
    'haut_tot',
    'haut_tronc',
    'tronc_diam',
    'fk_stadedev',
    'fk_pied',
    'fk_situation',
    'nomfrancais',
    'feuillage',
    'remarquable',
    'age_estim',
]


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    replacements = {
        'ras': np.nan,
        'inconnu': np.nan,
        'date inconnu': np.nan,
        'revetement non permeable': 'revetement non permeable',
    }
    return replacements.get(value, value)

text_columns = [
    'clc_quartier', 'clc_secteur', 'fk_arb_etat', 'fk_stadedev', 'fk_pied',
    'fk_situation', 'nomfrancais', 'feuillage', 'remarquable'
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(normalize_text)

numeric_columns = ['X', 'Y', 'haut_tot', 'haut_tronc', 'tronc_diam', 'age_estim']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['haut_tot', 'haut_tronc', 'tronc_diam', 'age_estim']:
    df.loc[df[col] <= 0, col] = np.nan

df = df.dropna(subset=[TARGET_SOURCE]).copy()
df[TARGET_SOURCE] = df[TARGET_SOURCE].astype(str)

df[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False)


In [ ]:
print(df[TARGET_SOURCE].value_counts())
plt.figure(figsize=(10, 4))
sns.countplot(x=TARGET_SOURCE, data=df, order=df[TARGET_SOURCE].value_counts().index)
plt.xticks(rotation=30, ha='right')
plt.title('Repartition de la cible fk_arb_etat')
plt.tight_layout()
plt.show()


In [ ]:
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_SOURCE].copy()

numeric_features = ['X', 'Y', 'haut_tot', 'haut_tronc', 'tronc_diam', 'age_estim']
categorical_features = [col for col in FEATURE_COLUMNS if col not in numeric_features]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=3000, class_weight='balanced')),
])

param_grid = [
    {
        'classifier': [LogisticRegression(max_iter=3000, class_weight='balanced')],
        'classifier__C': [0.1, 1.0, 5.0],
    },
    {
        'classifier': [RandomForestClassifier(class_weight='balanced', random_state=42)],
        'classifier__n_estimators': [200, 400],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_leaf': [1, 2, 5],
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print('Meilleur score CV F1 weighted :', grid.best_score_)
print('Meilleurs parametres :', grid.best_params_)


In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)
classes = list(best_model.classes_)
alert_probability = 1 - y_proba[:, classes.index(SAFE_STATE)] if SAFE_STATE in classes else None

serializable_best_params = {}
for key, value in grid.best_params_.items():
    if hasattr(value, 'get_params'):
        serializable_best_params[key] = value.__class__.__name__
    else:
        serializable_best_params[key] = value

metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_test, y_pred)),
    'precision_weighted': float(precision_score(y_test, y_pred, average='weighted', zero_division=0)),
    'recall_weighted': float(recall_score(y_test, y_pred, average='weighted', zero_division=0)),
    'f1_weighted': float(f1_score(y_test, y_pred, average='weighted', zero_division=0)),
    'roc_auc_ovr_weighted': float(roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted', labels=classes)),
    'target': TARGET_SOURCE,
    'safe_state': SAFE_STATE,
    'classes': classes,
    'best_params': serializable_best_params,
}

if alert_probability is not None:
    metrics['mean_alert_probability_on_test'] = float(np.mean(alert_probability))

metrics


In [ ]:
report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
print(classification_report(y_test, y_pred, zero_division=0))

labels = list(best_model.classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=30, colorbar=False)
plt.title('Matrice de confusion')
plt.tight_layout()
plt.show()

support_by_class = pd.Series({label: int(report_dict[label]['support']) for label in labels})
plt.figure(figsize=(8, 4))
sns.barplot(x=support_by_class.index, y=support_by_class.values)
plt.xticks(rotation=30, ha='right')
plt.ylabel("Nombre d'arbres")
plt.title("Repartition des arbres par classe dans l'ensemble de test")
plt.tight_layout()
plt.show()


In [ ]:
model_path = OUTPUT_DIR / 'bc3_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

example_row = X_test.dropna().iloc[0] if not X_test.dropna().empty else X_test.iloc[0].fillna('inconnu')
example_input = example_row.to_dict()
for key, value in example_input.items():
    if pd.isna(value):
        example_input[key] = None

print('Modele genere :')
print(model_path)
print('\nExemple de variables d\'entree pour tester le script final :')
print(example_input)
print('\nExemple de commande :')
example_command = ['python', 'predict_bc3.py', '--model_path', 'bc3_model.pkl']
for column in FEATURE_COLUMNS:
    example_command.extend([f'--{column}', str(example_input[column])])
print(' '.join(example_command))
